# LaCAM centralized baseline ( S3) — Colab/Kaggle

Runs the **real LaCAM** solver (Kei18/lacam3, via the official
`Cognitive-AI-Systems/pogema-benchmark` integration) on our den520d saturation
densities — a centralized planner for context. CPU-only; no GPU needed.

**Resumable & Drive-backed.** The sweep runs one `(agents, seed)` at a time and writes
its result to **Google Drive immediately**, skipping anything already done. A
disconnect costs at most the one episode in flight — just re-run the sweep cell to
continue. Nothing is recomputed.

**Order:** run top to bottom. The C++ build (cell 2) and the smoke test (cell 6) are
the gates — confirm each before moving on. Why a separate notebook: pogema-benchmark's
`eval.py` imports *every* solver; we import **only LaCAM** + their `create_env` and run
it through the same `pogema_toolbox.evaluation()` we use for Follower (comparable tp).

## 1. System build tools + clone repos

In [ ]:
!apt-get -qq update && apt-get -qq install -y cmake build-essential >/dev/null
import os
if not os.path.exists('/content/pogema-benchmark'):
    !git clone --recursive -q https://github.com/Cognitive-AI-Systems/pogema-benchmark.git /content/pogema-benchmark
!cd /content/pogema-benchmark && git submodule update --init --recursive
if not os.path.exists('/content/mapf-deadlock'):
    !git clone -q --depth 1 https://github.com/tay805/mapf-deadlock.git /content/mapf-deadlock
print('lacam3 present:', os.path.isdir('/content/pogema-benchmark/algorithms/lacam/lacam3'))

## 2. Build `liblacam.so`  *(gate: confirm the path prints)*

In [ ]:
%cd /content/pogema-benchmark/algorithms/lacam
!cmake -B build -DCMAKE_BUILD_TYPE=Release >/tmp/cmake.log 2>&1 && cmake --build build -j 2 >>/tmp/cmake.log 2>&1; tail -5 /tmp/cmake.log
import glob, shutil
so = glob.glob('/content/pogema-benchmark/algorithms/lacam/**/liblacam.so', recursive=True)
print('built:', so)
assert so, 'liblacam.so not built — see /tmp/cmake.log'
shutil.copy(so[0], '/content/pogema-benchmark/algorithms/lacam/liblacam.so')
print('placed at algorithms/lacam/liblacam.so')

## 3. Python env (py3.10 conda) + LaCAM's deps
Same py3.10 `follower` conda env as `colab_baseline.ipynb` (Colab's native 3.12 breaks
the pinned MAPF stack); reused if it already exists this session. **Kaggle:** already
py3.10 — set `USE_CONDA=False` to pip-install into the kernel and skip condacolab.

In [ ]:
import os, subprocess
USE_CONDA = True   # Colab: True. Kaggle (py3.10): False.
PKGS = 'pogema==1.3.0 pogema-toolbox==0.1.0 numpy<=1.23.1 pandas<=1.4 pyyaml dask==2024.8.0 distributed==2024.8.0 loguru cppimport pybind11==2.13.1 tabulate'
if USE_CONDA:
    try:
        import condacolab; condacolab.check()
    except Exception:
        !pip install -q condacolab
        import condacolab; condacolab.install()   # kernel restarts; re-run this cell after
    if 'follower' not in subprocess.run(['conda','env','list'],capture_output=True,text=True).stdout:
        !conda create -y -n follower python=3.10 >/dev/null
    !conda run -n follower pip install -q --prefer-binary {PKGS}
    PYRUN = 'conda run --no-capture-output -n follower python'
else:
    !pip install -q --prefer-binary {PKGS}
    PYRUN = 'python'
print('env ready; PYRUN =', PYRUN)

## 4. Mount Drive (results persist here)  +  scenario map & per-instance runner
Set `RESULTS` to your Drive folder. **Kaggle:** skip the mount and set
`RESULTS='/kaggle/working/mapf-deadlock-results'` (that path persists across the session).

In [ ]:
import os, yaml
# --- Drive (Colab) ---
try:
    from google.colab import drive; drive.mount('/content/gdrive')
    RESULTS = '/content/gdrive/MyDrive/mapf-deadlock-results'   # <-- adjust to your Drive path
except Exception:
    RESULTS = '/kaggle/working/mapf-deadlock-results'           # Kaggle fallback (persists)
os.makedirs(RESULTS, exist_ok=True)
print('RESULTS =', RESULTS)
# --- den520d map (from our repo) into the benchmark's experiments tree ---
EXP = '/content/pogema-benchmark/algorithms/experiments/99-den520d-sat'
os.makedirs(EXP, exist_ok=True)
ours = yaml.safe_load(open('/content/mapf-deadlock/learn-to-follow/env/test-maps.yaml'))
yaml.safe_dump({'den520d': ours['den520d']}, open(f'{EXP}/maps.yaml','w'))
# --- per-instance runner: ONE (agents, seed) -> save JSON to <out> ---
runner = '''import sys
from pathlib import Path
import yaml
from pogema_toolbox.evaluator import evaluation
from pogema_toolbox.registry import ToolboxRegistry
from create_env import create_env_base
from pogema_toolbox.create_env import Environment
from lacam.inference import LacamInference, LacamInferenceConfig
N, seed, out, steps = int(sys.argv[1]), int(sys.argv[2]), sys.argv[3], int(sys.argv[4])
ToolboxRegistry.register_env("Environment", create_env_base, Environment)
ToolboxRegistry.register_algorithm("LaCAM", LacamInference, LacamInferenceConfig)
ToolboxRegistry.register_maps(yaml.safe_load(open("experiments/99-den520d-sat/maps.yaml")))
cfg = {"environment": {"name":"Environment","collision_system":"soft","on_target":"restart",
        "observation_type":"POMAPF","max_episode_steps":steps,"map_name":"den520d",
        "num_agents":{"grid_search":[N]},"seed":{"grid_search":[seed]}},
       "algorithms":{"LaCAM":{"name":"LaCAM","num_process":1,"parallel_backend":"sequential"}}}
Path(out).mkdir(parents=True, exist_ok=True)
evaluation(cfg, eval_dir=Path(out))
print("DONE", out)
'''
open('/content/pogema-benchmark/algorithms/run_lacam.py','w').write(runner)
print('wrote maps.yaml + run_lacam.py')

## 5. Smoke test — 1 instance (128 agents, seed 0, 128 steps)  *(gate)*

In [ ]:
%cd /content/pogema-benchmark/algorithms
!{PYRUN} run_lacam.py 128 0 /tmp/lacam_smoke 128
import json, glob
j = glob.glob('/tmp/lacam_smoke/*.json')
print('result files:', j)
assert j, 'no result — check the error above before running the sweep'
print('smoke throughput:', json.load(open(j[0]))[0]['metrics'].get('avg_throughput'))

## 6. Full sweep — den520d 128–640 × 3 seeds  *(resumable, saves to Drive)*
One `(agents, seed)` at a time, written straight to `RESULTS` and skipped if already
there. Re-run freely after a disconnect — it resumes. (640 agents can be slow per
LaCAM replan; each instance is saved the moment it finishes.)

In [ ]:
import os, subprocess, time
RES = f'{RESULTS}/lacam-sat'
DENS, SEEDS = [128, 256, 384, 512, 640], [0, 1, 2]
os.chdir('/content/pogema-benchmark/algorithms')
for N in DENS:
    for k in SEEDS:
        out = f'{RES}/n{N}_s{k}'
        if os.path.exists(f'{out}/LaCAM.json'):
            print(f'n{N} s{k}: done, skip', flush=True); continue
        print(f'=== n{N} s{k} ===', flush=True); t0 = time.time()
        r = subprocess.run(PYRUN.split() + ['run_lacam.py', str(N), str(k), out, '512'])
        ok = os.path.exists(f'{out}/LaCAM.json')
        print(f"    {'OK' if ok else f'FAILED rc={r.returncode}'} in {(time.time()-t0)/60:.1f} min", flush=True)
print('done — run the results cell.')

## 7. LaCAM throughput vs Follower / density control

In [ ]:
import json, glob
from collections import defaultdict
RES = f'{RESULTS}/lacam-sat'
agg = defaultdict(list)
for fp in glob.glob(f'{RES}/n*_s*/LaCAM.json'):
    for r in json.load(open(fp)):
        agg[r['env_grid_search']['num_agents']].append(r['metrics']['avg_throughput'])
ref = {128:2.23, 256:2.32, 384:1.92, 512:1.56, 640:1.06}   # Follower all-active (3 seeds)
print(f"{'agents':>6}{'LaCAM':>9}{'n':>3}{'Follower':>10}")
for n in sorted(agg):
    x = agg[n]; print(f"{n:>6}{sum(x)/len(x):>9.3f}{len(x):>3}{ref.get(n,0):>10.2f}")
print('\nDensity control (den520d hide-meter): ~192 active -> ~2.47 with 640 present.')